In [1]:
import os
import sys
os.environ["KERAS_BACKEND"] = "torch"
sys.path.append("./src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.model_selection import train_test_split
import keras
from keras.metrics import Recall, Precision
from sklearn.metrics import confusion_matrix
from keras.callbacks import ModelCheckpoint
from keras.models import load_model

from models import *
from utils import *


# -------------------
# CONFIG
# -------------------
DATA_ROOT = "processed_data"
Multi_scale_patch = False
IMAGE_FOLDER = os.path.join(DATA_ROOT, "images")
LABELS_FILE = os.path.join(DATA_ROOT, "labels.csv")
MODEL_DIR = "models"

CONFIG = {
    "img_size": (255, 255, 1),
    "batch_size": 8,
    "epochs": 20,
    "learning_rate": 1e-5,
    "model_name": "resnet18",   # 👈 change this to switch model
}


In [2]:
list_models()

Available models:
- baseline
- deeper
- deeperv2
- deeperv3
- FCN
- multiscale
- resnet50
- resnet18
- mobilenetv2_transfer


In [3]:
# -------------------
# HELPERS
# -------------------
def compile_model(model, config):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config["learning_rate"]),
        loss='binary_crossentropy',
        metrics=['accuracy', Recall(name="recall"), Precision(name="precision")]
    )
    return model

def get_model_path(config):
    model_name = config["model_name"]
    
    
    ps = config["img_size"][0]


    filename = f"{model_name}_ps{ps}.keras"
    return os.path.join(MODEL_DIR, filename)

In [4]:
# -------------------
# LOAD LABELS
# -------------------
df = pd.read_csv(LABELS_FILE)

print("Total samples:", len(df))
print(df['label'].value_counts())

# -------------------
# SPLIT
# -------------------
train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

# -------------------
# LOAD INTO MEMORY
# -------------------
X_train, Y_train = load_data(train_df, DATA_ROOT, Multi_scale_patch)
X_val, Y_val = load_data(val_df, DATA_ROOT, Multi_scale_patch)
X_test, Y_test = load_data(test_df, DATA_ROOT, Multi_scale_patch)


if Multi_scale_patch:
    print("Train balance:", np.mean(Y_train))
    print("Train:", X_train[0].shape)
    print("Train:", X_train[1].shape)
    print("Val:", X_val[0].shape)
    print("Val:", X_val[1].shape)
    print("Test:", X_test[0].shape)
    print("Test:", X_test[1].shape)
else:
    print("Train:", X_train.shape)
    print("Val:", X_val.shape)
    print("Test:", X_test.shape)


Total samples: 2632
label
1    1316
0    1316
Name: count, dtype: int64
Train: (1842, 255, 255, 1)
Val: (395, 255, 255, 1)
Test: (395, 255, 255, 1)


In [5]:
# -------------------
# Model training
# -------------------
# build model
model = get_model(CONFIG["model_name"],CONFIG["img_size"]) 

# compile
model = compile_model(model, CONFIG)

model.summary()

model_path = get_model_path(CONFIG)

checkpoint = ModelCheckpoint(
    filepath=model_path,
    monitor='val_loss',
    save_best_only=True,
    mode='min',
    verbose=1
)

# train
if Multi_scale_patch:
    history = model.fit(
        [X_train[0],X_train[1]], Y_train,
        validation_data=([X_val[0],X_val[1]], Y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        callbacks=[checkpoint]
    )
else:
    history = model.fit(
    X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=CONFIG["epochs"],
        batch_size=CONFIG["batch_size"],
        callbacks=[checkpoint]
    )

print("Loading best model from:", model_path)
model = load_model(model_path)

# -------------------
# PREDICTIONS
# -------------------
if Multi_scale_patch:
    y_pred_probs = model.predict([X_test[0], X_test[1]])
else:
    y_pred_probs = model.predict(X_test)

# -------------------
# THRESHOLD
# -------------------
threshold = 0.5
y_pred = (y_pred_probs > threshold).astype(int).flatten()
y_true = Y_test.astype(int)

# -------------------
# PER-LABEL ACCURACY
# -------------------
for label in [0, 1]:
    idx = (y_true == label)
    
    if np.sum(idx) == 0:
        print(f"Label {label}: no samples")
        continue

    acc = np.mean(y_pred[idx] == y_true[idx])
    print(f"Accuracy for label {label}: {acc:.4f} (n={np.sum(idx)})")

cm = confusion_matrix(y_true, y_pred)


# pretty print
print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print(f"\nTrue Negatives: {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives: {tp}")
# test
model.evaluate(X_test, Y_test)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 255, 255,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 255, 255,  │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 255, 255,  │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 255, 255,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 255, 255,  │     36,928 │ activation[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 255, 255,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 255, 255,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 255, 255,  │     36,928 │ activation_1[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 255, 255,  │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 255, 255,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 255, 255,  │          0 │ add[0][0]         │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 255, 255,  │     36,928 │ activation_2[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 255, 255,  │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 255, 255,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 255, 255,  │     36,928 │ activation_3[0][… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 255, 255,  │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 255, 255,  │          0 │ batch_normalizat

 Total params: 11,247,873 (42.91 MB)

 Trainable params: 11,238,273 (42.87 MB)

 Non-trainable params: 9,600 (37.50 KB)

Epoch 1/20
 59/231 ━━━━━━━━━━━━━━━━━━━━ 20s 122ms/step - accuracy: 0.7021 - loss: 0.6208 - precision: 0.6751 - recall: 0.8305

KeyboardInterrupt: 